# A/B Testing: Checkout Button Experiment
## Es real la mejora o es ruido estadistico?

Experimento A/B en e-commerce: boton de checkout Control vs Treatment.  
Proyecto de [@aroaxinping](https://tiktok.com/@aroaxinping) — estudiante de Data Science.

---
## 0. Configuracion del entorno

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# --- Tema oscuro personalizado ---
BG_DARK   = '#0f0f0f'
ORANGE    = '#e85d04'
BLUE      = '#6a9ad4'
WHITE     = '#f0f0f0'
GRAY      = '#888888'
GREEN     = '#2dc653'
RED       = '#e74c3c'
PURPLE    = '#9b59b6'

mpl.rcParams.update({
    'figure.facecolor': BG_DARK,
    'axes.facecolor':   BG_DARK,
    'axes.edgecolor':   GRAY,
    'axes.labelcolor':  WHITE,
    'text.color':       WHITE,
    'xtick.color':      GRAY,
    'ytick.color':      GRAY,
    'grid.color':       '#2a2a2a',
    'grid.alpha':       0.6,
    'figure.figsize':   (12, 5),
    'font.size':        11,
    'font.family':      'monospace',
    'axes.grid':        True,
})

print("Entorno listo.")

---
## 1. Datos

Cargamos los datos del experimento A/B desde `data/processed/ab_test_data.csv`.  
Si no existen, generamos un dataset sintetico de ~20,000 usuarios.

In [ ]:
# -----------------------------------------------------------------------
# Intentar cargar datos procesados
# Si no existen, generar sinteticos
# -----------------------------------------------------------------------
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

from pathlib import Path

DATA_PATH = Path("../data/processed/ab_test_data.csv")
SYNTHETIC = False

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
    print(f"Datos cargados: {len(df)} usuarios")
else:
    print("No hay datos procesados. Generando dataset sintetico...")
    from fetch_ab_data import generate_ab_data
    df = generate_ab_data()
    SYNTHETIC = True

print(f"\nColumnas: {df.columns.tolist()}")
print(f"Grupos: {df['group'].value_counts().to_dict()}")
df.head()

---
## 2. Exploracion

> **Pregunta:** Los dos grupos son comparables antes de analizar el resultado?

Antes de mirar la conversion, verificamos que la aleatorizacion funciono:
los dos grupos deben ser similares en device, country y session duration.

In [ ]:
# -----------------------------------------------------------------------
# 2a. Sample size por grupo
# -----------------------------------------------------------------------
print("=== Tamano de muestra ===\n")
print(df['group'].value_counts())
print(f"\nTotal: {len(df)} usuarios")

In [ ]:
# -----------------------------------------------------------------------
# 2b. Balance check: device distribution
# -----------------------------------------------------------------------
device_balance = pd.crosstab(df['group'], df['device'], normalize='index') * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Device distribution
device_balance.plot(kind='bar', ax=axes[0], color=[ORANGE, BLUE], alpha=0.85, edgecolor=BG_DARK)
axes[0].set_title('Distribucion de device por grupo (%)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Porcentaje')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].legend(title='Device')

# Country distribution
country_balance = pd.crosstab(df['group'], df['country'], normalize='index') * 100
country_balance.plot(kind='bar', ax=axes[1], color=[ORANGE, BLUE, GREEN, RED, PURPLE, GRAY, WHITE], alpha=0.85, edgecolor=BG_DARK)
axes[1].set_title('Distribucion de country por grupo (%)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Porcentaje')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(title='Country', fontsize=8)

plt.tight_layout()
plt.show()

print("\nDevice balance:")
print(device_balance.round(1))
print("\nCountry balance:")
print(country_balance.round(1))

In [ ]:
# -----------------------------------------------------------------------
# 2c. Session duration por grupo (sanity check)
# -----------------------------------------------------------------------
# NOTA: session_duration es una variable POST-tratamiento. Los usuarios que
# convierten tienden a tener sesiones mas largas, por lo que session_duration
# esta afectada por el outcome (conversion). Esto NO es un balance check
# pre-experimento — solo es una descripcion de la distribucion observada.
# Un verdadero balance check usaria solo variables pre-asignacion (device,
# country, timestamp de entrada, etc.).

fig, ax = plt.subplots(figsize=(12, 5))

for grp, color in [('control', BLUE), ('treatment', ORANGE)]:
    data = df[df['group'] == grp]['session_duration_sec']
    ax.hist(data, bins=50, alpha=0.6, color=color, label=f'{grp} (media={data.mean():.0f}s)', edgecolor=BG_DARK)

ax.set_xlabel('Session duration (segundos)')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribucion de session duration por grupo', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

# Test de balance
stat, p = stats.mannwhitneyu(
    df[df['group'] == 'control']['session_duration_sec'],
    df[df['group'] == 'treatment']['session_duration_sec'],
)
print(f"Mann-Whitney U (session duration): U={stat:.0f}, p={p:.4f}")
print(f"{'Los grupos son comparables en session duration.' if p > 0.05 else 'ATENCION: diferencia significativa en session duration.'}")
print("\nCAVEAT: session_duration es post-treatment (afectada por conversion),")
print("por lo que esta comparacion NO es un balance check pre-experimento valido.")

In [ ]:
# -----------------------------------------------------------------------
# 2d. Conversion rate cruda por grupo
# -----------------------------------------------------------------------
conversion = df.groupby('group')['converted'].agg(['sum', 'count', 'mean'])
conversion.columns = ['conversiones', 'usuarios', 'tasa_conversion']
conversion['tasa_conversion_pct'] = conversion['tasa_conversion'] * 100

print("=== Conversion rate por grupo ===\n")
print(conversion.to_string())

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    conversion.index,
    conversion['tasa_conversion_pct'],
    color=[BLUE, ORANGE],
    alpha=0.85,
    edgecolor=BG_DARK,
    width=0.5,
)
for bar, val in zip(bars, conversion['tasa_conversion_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f}%', ha='center', fontweight='bold', color=WHITE, fontsize=13)

ax.set_ylabel('Conversion rate (%)')
ax.set_title('Conversion rate: Control vs Treatment', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(conversion['tasa_conversion_pct']) * 1.3)
plt.tight_layout()
plt.show()

diff_pct = conversion.loc['treatment', 'tasa_conversion_pct'] - conversion.loc['control', 'tasa_conversion_pct']
lift = diff_pct / conversion.loc['control', 'tasa_conversion_pct'] * 100
print(f"\nDiferencia absoluta: {diff_pct:.2f} pp")
print(f"Lift relativo: {lift:.1f}%")

---
## 3. Test de hipotesis

> **Pregunta:** La diferencia en conversion es estadisticamente significativa o es ruido?

- **H0**: p_treatment = p_control (no hay diferencia)
- **H1**: p_treatment != p_control (hay diferencia)
- alpha = 0.05

In [ ]:
# -----------------------------------------------------------------------
# 3a. Z-test para proporciones
# -----------------------------------------------------------------------
control = df[df['group'] == 'control']
treatment = df[df['group'] == 'treatment']

# Conversiones y tamanos
successes = np.array([treatment['converted'].sum(), control['converted'].sum()])
nobs = np.array([len(treatment), len(control)])

# Z-test two-sided
z_stat, p_value = proportions_ztest(successes, nobs, alternative='two-sided')

print("=== Z-test para proporciones ===\n")
print(f"  Conversiones treatment: {successes[0]} / {nobs[0]}")
print(f"  Conversiones control:   {successes[1]} / {nobs[1]}")
print(f"\n  Z-statistic: {z_stat:.4f}")
print(f"  p-value:     {p_value:.6f}")
print(f"\n  alpha = 0.05")

if p_value < 0.05:
    print(f"  >>> Rechazamos H0. La diferencia ES estadisticamente significativa.")
else:
    print(f"  >>> No rechazamos H0. La diferencia NO es significativa.")

In [ ]:
# -----------------------------------------------------------------------
# 3b. Intervalo de confianza para la diferencia
# -----------------------------------------------------------------------
p_treat = treatment['converted'].mean()
p_ctrl = control['converted'].mean()
n_treat = len(treatment)
n_ctrl = len(control)

# SE de la diferencia de proporciones
se_diff = np.sqrt(p_treat * (1 - p_treat) / n_treat + p_ctrl * (1 - p_ctrl) / n_ctrl)

# IC 95%
z_crit = 1.96
diff = p_treat - p_ctrl
ci_lower = diff - z_crit * se_diff
ci_upper = diff + z_crit * se_diff

print("=== Intervalo de confianza (95%) para la diferencia ===\n")
print(f"  p_treatment - p_control = {diff*100:.3f} pp")
print(f"  IC 95%: [{ci_lower*100:.3f}, {ci_upper*100:.3f}] pp")
print(f"\n  {'El IC no incluye 0 -> diferencia significativa' if ci_lower > 0 else 'El IC incluye 0 -> no significativa'}")

# Visualizacion
fig, ax = plt.subplots(figsize=(10, 3))
ax.errorbar(diff * 100, 0, xerr=z_crit * se_diff * 100,
            fmt='o', color=ORANGE, markersize=10, capsize=8, capthick=2, linewidth=2)
ax.axvline(0, color=WHITE, linestyle='--', alpha=0.5, label='Sin efecto')
ax.set_xlabel('Diferencia en conversion (pp)')
ax.set_title('Intervalo de confianza 95% para la diferencia', fontsize=14, fontweight='bold')
ax.set_yticks([])
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Tamano de muestra

> **Pregunta:** Cuantos usuarios necesitabamos para detectar esta diferencia?

Power analysis: dado el effect size observado, calculamos el tamano de muestra
necesario para detectar la diferencia con 80% de poder estadistico.

In [ ]:
# -----------------------------------------------------------------------
# 4a. Power analysis
# -----------------------------------------------------------------------
# Effect size (Cohen's h) para proporciones
es = proportion_effectsize(p_ctrl, p_treat)
print(f"Effect size (Cohen's h): {es:.4f}")

# Tamano de muestra necesario para 80% power
power_analysis = NormalIndPower()
required_n = power_analysis.solve_power(
    effect_size=abs(es),
    alpha=0.05,
    power=0.80,
    alternative='two-sided',
)
print(f"Tamano de muestra necesario por grupo (80% power): {required_n:.0f}")
print(f"Tamano de muestra que tenemos por grupo: {n_ctrl}")
print(f"\n{'Nuestro experimento ESTA bien powered.' if n_ctrl >= required_n else 'Nuestro experimento NO tiene suficiente poder.'}")

# Power real con nuestro n
actual_power = power_analysis.solve_power(
    effect_size=abs(es),
    nobs1=n_ctrl,
    alpha=0.05,
    alternative='two-sided',
)
print(f"Power real con n={n_ctrl}: {actual_power:.1%}")

In [ ]:
# -----------------------------------------------------------------------
# 4b. Plot: required sample size vs detectable effect size
# -----------------------------------------------------------------------
effect_sizes = np.linspace(0.01, 0.15, 50)
required_ns = [
    power_analysis.solve_power(effect_size=e, alpha=0.05, power=0.80, alternative='two-sided')
    for e in effect_sizes
]

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(effect_sizes, required_ns, color=ORANGE, linewidth=2.5)
ax.axhline(n_ctrl, color=BLUE, linestyle='--', alpha=0.7, label=f'Nuestro n={n_ctrl}')
ax.axvline(abs(es), color=GREEN, linestyle='--', alpha=0.7, label=f'Nuestro effect size={abs(es):.4f}')

ax.set_xlabel("Effect size (Cohen's h)")
ax.set_ylabel('Sample size por grupo')
ax.set_title('Tamano de muestra necesario vs effect size (power=80%, alpha=0.05)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, max(required_ns) * 1.1)
plt.tight_layout()
plt.show()

---
## 5. Segmentacion

> **Pregunta:** El efecto es igual en mobile y desktop? Y por pais?

Cortamos los datos por device y country para ver si el treatment
tiene un efecto uniforme o varia por segmento.

In [ ]:
# -----------------------------------------------------------------------
# 5a. Conversion rate por grupo x device
# -----------------------------------------------------------------------
seg_device = df.groupby(['device', 'group'])['converted'].agg(['sum', 'count', 'mean']).reset_index()
seg_device['pct'] = seg_device['mean'] * 100

print("=== Conversion por device x grupo ===\n")
pivot_device = seg_device.pivot(index='device', columns='group', values='pct')
print(pivot_device.round(3))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(pivot_device.index))
width = 0.35
ax.bar(x - width/2, pivot_device['control'], width, label='Control', color=BLUE, alpha=0.85)
ax.bar(x + width/2, pivot_device['treatment'], width, label='Treatment', color=ORANGE, alpha=0.85)

for i, dev in enumerate(pivot_device.index):
    ax.text(i - width/2, pivot_device.loc[dev, 'control'] + 0.05,
            f"{pivot_device.loc[dev, 'control']:.2f}%", ha='center', color=WHITE, fontsize=10)
    ax.text(i + width/2, pivot_device.loc[dev, 'treatment'] + 0.05,
            f"{pivot_device.loc[dev, 'treatment']:.2f}%", ha='center', color=WHITE, fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(pivot_device.index)
ax.set_ylabel('Conversion rate (%)')
ax.set_title('Conversion rate por device y grupo', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------------------
# 5b. Conversion rate por grupo x country
# -----------------------------------------------------------------------
seg_country = df.groupby(['country', 'group'])['converted'].agg(['sum', 'count', 'mean']).reset_index()
seg_country['pct'] = seg_country['mean'] * 100

pivot_country = seg_country.pivot(index='country', columns='group', values='pct')
print("=== Conversion por country x grupo ===\n")
print(pivot_country.round(3))

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(pivot_country.index))
width = 0.35
ax.bar(x - width/2, pivot_country['control'], width, label='Control', color=BLUE, alpha=0.85)
ax.bar(x + width/2, pivot_country['treatment'], width, label='Treatment', color=ORANGE, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(pivot_country.index)
ax.set_ylabel('Conversion rate (%)')
ax.set_title('Conversion rate por country y grupo', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------------------
# 5c. Chi-square test per segment
# -----------------------------------------------------------------------
print("=== Chi-square test por segmento ===\n")

segment_results = []

# Por device
for device in df['device'].unique():
    subset = df[df['device'] == device]
    table = pd.crosstab(subset['group'], subset['converted'])
    chi2, p, dof, expected = stats.chi2_contingency(table)
    result = {
        'segmento': f'device={device}',
        'chi2': chi2,
        'p_value': p,
        'significativo': 'Si' if p < 0.05 else 'No',
    }
    segment_results.append(result)
    print(f"  {device}: chi2={chi2:.4f}, p={p:.4f} {'*' if p < 0.05 else ''}")

print()

# Por country
for country in sorted(df['country'].unique()):
    subset = df[df['country'] == country]
    table = pd.crosstab(subset['group'], subset['converted'])
    chi2, p, dof, expected = stats.chi2_contingency(table)
    result = {
        'segmento': f'country={country}',
        'chi2': chi2,
        'p_value': p,
        'significativo': 'Si' if p < 0.05 else 'No',
    }
    segment_results.append(result)
    print(f"  {country}: chi2={chi2:.4f}, p={p:.4f} {'*' if p < 0.05 else ''}")

seg_results_df = pd.DataFrame(segment_results)
print("\n")
print(seg_results_df.to_string(index=False))

In [ ]:
# -----------------------------------------------------------------------
# 5d. Interaction plot: grupo x device
# -----------------------------------------------------------------------
interaction = df.groupby(['device', 'group'])['converted'].mean().reset_index()
interaction['pct'] = interaction['converted'] * 100

fig, ax = plt.subplots(figsize=(10, 6))

for grp, color, marker in [('control', BLUE, 's'), ('treatment', ORANGE, 'o')]:
    data = interaction[interaction['group'] == grp]
    ax.plot(data['device'], data['pct'], marker=marker, color=color,
            linewidth=2.5, markersize=10, label=grp)

ax.set_xlabel('Device')
ax.set_ylabel('Conversion rate (%)')
ax.set_title('Interaction plot: Grupo x Device', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print("Si las lineas no son paralelas, hay interaccion entre grupo y device.")
print("El treatment podria funcionar mejor en mobile que en desktop (o viceversa).")

---
## 6. Metricas secundarias

> **Pregunta:** El nuevo boton tambien cambia el tiempo de sesion?

Ademas de la conversion, analizamos si el treatment afecta session duration.
Usamos Mann-Whitney U (no parametrico, robusto a distribuciones no normales).

In [ ]:
# -----------------------------------------------------------------------
# 6a. Session duration: control vs treatment
# -----------------------------------------------------------------------
dur_ctrl = control['session_duration_sec']
dur_treat = treatment['session_duration_sec']

print("=== Session duration por grupo ===\n")
print(f"  Control:   media={dur_ctrl.mean():.1f}s, mediana={dur_ctrl.median():.1f}s, std={dur_ctrl.std():.1f}s")
print(f"  Treatment: media={dur_treat.mean():.1f}s, mediana={dur_treat.median():.1f}s, std={dur_treat.std():.1f}s")

# Mann-Whitney U test
u_stat, u_pvalue = stats.mannwhitneyu(dur_treat, dur_ctrl, alternative='two-sided')
print(f"\n  Mann-Whitney U: U={u_stat:.0f}, p={u_pvalue:.6f}")
if u_pvalue < 0.05:
    print("  >>> Diferencia significativa en session duration.")
else:
    print("  >>> No hay diferencia significativa en session duration.")

In [ ]:
# -----------------------------------------------------------------------
# 6b. Distribution comparison plot
# -----------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma superpuesto
for grp, color, label in [('control', BLUE, 'Control'), ('treatment', ORANGE, 'Treatment')]:
    data = df[df['group'] == grp]['session_duration_sec']
    axes[0].hist(data, bins=50, alpha=0.5, color=color, label=label, edgecolor=BG_DARK)

axes[0].set_xlabel('Session duration (s)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribucion de session duration', fontsize=14, fontweight='bold')
axes[0].legend()

# Box plot
bp = axes[1].boxplot(
    [dur_ctrl, dur_treat],
    labels=['Control', 'Treatment'],
    patch_artist=True,
    boxprops=dict(facecolor=BG_DARK, color=WHITE),
    medianprops=dict(color=ORANGE, linewidth=2),
    whiskerprops=dict(color=GRAY),
    capprops=dict(color=GRAY),
    flierprops=dict(marker='o', markerfacecolor=GRAY, markersize=3, alpha=0.3),
)
bp['boxes'][0].set_facecolor(BLUE)
bp['boxes'][0].set_alpha(0.3)
bp['boxes'][1].set_facecolor(ORANGE)
bp['boxes'][1].set_alpha(0.3)

axes[1].set_ylabel('Session duration (s)')
axes[1].set_title('Box plot: session duration por grupo', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 7. Errores comunes en A/B Testing

> **Pregunta:** Que puede salir mal?

Tres trampas clasicas que arruinan un experimento A/B:
1. **Peeking problem** — mirar resultados demasiado pronto
2. **Multiple comparisons** — testear muchos segmentos sin corregir
3. **Simpson's paradox** — el resultado agregado contradice los segmentos

In [ ]:
# -----------------------------------------------------------------------
# 7a. Peeking problem: p-value over time
# -----------------------------------------------------------------------
# Simular que miramos el p-value cada dia del experimento
df_sorted = df.sort_values('timestamp').reset_index(drop=True)

check_points = np.linspace(100, len(df_sorted), 50, dtype=int)
p_values_over_time = []
n_samples_over_time = []

for n in check_points:
    subset = df_sorted.iloc[:n]
    ctrl_sub = subset[subset['group'] == 'control']
    treat_sub = subset[subset['group'] == 'treatment']

    if len(ctrl_sub) < 10 or len(treat_sub) < 10:
        continue

    succ = np.array([treat_sub['converted'].sum(), ctrl_sub['converted'].sum()])
    nob = np.array([len(treat_sub), len(ctrl_sub)])

    try:
        _, p = proportions_ztest(succ, nob, alternative='two-sided')
        p_values_over_time.append(p)
        n_samples_over_time.append(n)
    except Exception:
        continue

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(n_samples_over_time, p_values_over_time, color=ORANGE, linewidth=2)
ax.axhline(0.05, color=RED, linestyle='--', linewidth=2, label='alpha = 0.05')
ax.fill_between(n_samples_over_time, 0, 0.05, color=RED, alpha=0.1)

ax.set_xlabel('Usuarios acumulados')
ax.set_ylabel('p-value')
ax.set_title('Peeking problem: p-value a lo largo del experimento', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

n_early_sig = sum(1 for p in p_values_over_time[:len(p_values_over_time)//2] if p < 0.05)
print(f"Veces que el p-value cruzo 0.05 en la primera mitad: {n_early_sig}")
print("Si paramos el test en esos momentos, tomariamos decisiones con datos insuficientes.")
print("REGLA: definir el tamano de muestra ANTES y no mirar hasta completar.")

In [ ]:
# -----------------------------------------------------------------------
# 7b. Multiple comparisons (Bonferroni correction)
# -----------------------------------------------------------------------
print("=== Correccion de Bonferroni ===\n")
print("Cuando testeamos muchos segmentos, aumenta la probabilidad de")
print("encontrar un falso positivo por puro azar.\n")

# Numero de tests realizados en la seccion de segmentacion
n_tests = len(seg_results_df)
alpha_original = 0.05
alpha_bonferroni = alpha_original / n_tests

print(f"  Tests realizados: {n_tests}")
print(f"  alpha original: {alpha_original}")
print(f"  alpha Bonferroni: {alpha_bonferroni:.4f}")
print()

# Re-evaluar significancia con Bonferroni
seg_results_df['sig_bonferroni'] = seg_results_df['p_value'].apply(
    lambda p: 'Si' if p < alpha_bonferroni else 'No'
)

print("Resultados con correccion de Bonferroni:")
print(seg_results_df[['segmento', 'p_value', 'significativo', 'sig_bonferroni']].to_string(index=False))

n_orig_sig = (seg_results_df['significativo'] == 'Si').sum()
n_bonf_sig = (seg_results_df['sig_bonferroni'] == 'Si').sum()
print(f"\nSignificativos sin correccion: {n_orig_sig}")
print(f"Significativos con Bonferroni: {n_bonf_sig}")
if n_orig_sig > n_bonf_sig:
    print("Algunos resultados que parecian significativos no sobreviven la correccion.")

In [ ]:
# -----------------------------------------------------------------------
# 7c. Simpson's paradox demo
# -----------------------------------------------------------------------
print("=== Simpson's paradox demo ===\n")
print("Simpson's paradox: una tendencia que aparece en los datos agregados")
print("se invierte cuando separamos por segmento.\n")

# Conversion global
print("RESULTADO GLOBAL:")
print(f"  Control:   {p_ctrl*100:.2f}%")
print(f"  Treatment: {p_treat*100:.2f}%")
overall_winner = 'Treatment' if p_treat > p_ctrl else 'Control'
print(f"  -> {overall_winner} gana por {abs(p_treat - p_ctrl)*100:.2f} pp")

# Conversion por device — mostrar la reversal
print("\nRESULTADO POR DEVICE:")
paradox_found = False
device_winners = {}

for device in ['desktop', 'mobile']:
    ctrl_dev = df[(df['group'] == 'control') & (df['device'] == device)]
    treat_dev = df[(df['group'] == 'treatment') & (df['device'] == device)]
    p_c = ctrl_dev['converted'].mean()
    p_t = treat_dev['converted'].mean()
    n_c = len(ctrl_dev)
    n_t = len(treat_dev)
    diff_pp = (p_t - p_c) * 100
    winner = 'Treatment' if diff_pp > 0 else 'Control'
    device_winners[device] = winner
    print(f"  {device}: Control={p_c*100:.2f}% (n={n_c}), Treatment={p_t*100:.2f}% (n={n_t}) -> {winner} ({diff_pp:+.2f} pp)")

# Detectar si hay paradoja real
if len(set(device_winners.values())) > 1 or (overall_winner not in device_winners.values()):
    paradox_found = True

# Mostrar distribucion de device por grupo (la clave de la paradoja)
print("\nDISTRIBUCION DE DEVICE POR GRUPO:")
for grp in ['control', 'treatment']:
    grp_data = df[df['group'] == grp]
    desk_pct = (grp_data['device'] == 'desktop').mean() * 100
    mob_pct = (grp_data['device'] == 'mobile').mean() * 100
    print(f"  {grp}: desktop={desk_pct:.1f}%, mobile={mob_pct:.1f}%")

if paradox_found:
    print("\n** SIMPSON'S PARADOX DETECTADO **")
    print("El resultado GLOBAL favorece a un grupo, pero al segmentar por device")
    print("la direccion cambia en al menos un segmento.")
    print("Esto ocurre porque la proporcion de desktop/mobile NO es igual en ambos grupos.")
    print("El grupo con mas usuarios en el segmento de alta conversion domina el agregado.")
else:
    print("\nEn estos datos no se observa una reversal completa por device,")
    print("pero en experimentos reales con asignacion imperfecta es un riesgo real.")

# Visualizacion de la paradoja
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: conversion por device x grupo
for grp, color in [('control', BLUE), ('treatment', ORANGE)]:
    rates = []
    for device in ['desktop', 'mobile']:
        subset = df[(df['group'] == grp) & (df['device'] == device)]
        rates.append(subset['converted'].mean() * 100)
    axes[0].plot(['desktop', 'mobile'], rates, marker='o', color=color,
                 linewidth=2.5, markersize=10, label=grp)

axes[0].set_ylabel('Conversion rate (%)')
axes[0].set_title("Conversion por device (la reversal)", fontsize=14, fontweight='bold')
axes[0].legend(fontsize=12)

# Panel 2: tamano de grupo por device (la causa)
grp_device = df.groupby(['group', 'device']).size().unstack(fill_value=0)
grp_device.plot(kind='bar', ax=axes[1], color=[ORANGE, BLUE], alpha=0.85, edgecolor=BG_DARK)
axes[1].set_title('N usuarios por grupo y device (el desbalance)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('N usuarios')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(title='Device')

plt.tight_layout()
plt.show()

---
## 8. Sintesis y conclusiones

In [ ]:
# -----------------------------------------------------------------------
# 8a. Summary table con todos los test results
# -----------------------------------------------------------------------
summary = pd.DataFrame([
    {
        'Test': 'Z-test conversion (global)',
        'Statistic': f'z={z_stat:.4f}',
        'p-value': f'{p_value:.6f}',
        'Significativo (alpha=0.05)': 'Si' if p_value < 0.05 else 'No',
    },
    {
        'Test': 'Mann-Whitney U (session duration)',
        'Statistic': f'U={u_stat:.0f}',
        'p-value': f'{u_pvalue:.6f}',
        'Significativo (alpha=0.05)': 'Si' if u_pvalue < 0.05 else 'No',
    },
])

# Anadir tests por segmento
for _, row in seg_results_df.iterrows():
    summary = pd.concat([summary, pd.DataFrame([{
        'Test': f"Chi2 ({row['segmento']})",
        'Statistic': f"chi2={row['chi2']:.4f}",
        'p-value': f"{row['p_value']:.6f}",
        'Significativo (alpha=0.05)': row['significativo'],
    }])], ignore_index=True)

print("=" * 70)
print("  RESUMEN DE TODOS LOS TESTS")
print("=" * 70)
print()
print(summary.to_string(index=False))
print()

In [ ]:
# -----------------------------------------------------------------------
# 8b. Decision: ship or not ship?
# -----------------------------------------------------------------------
print("=" * 70)
print("  DECISION FINAL")
print("=" * 70)

print(f"""
  Conversion control:   {p_ctrl*100:.2f}%
  Conversion treatment: {p_treat*100:.2f}%
  Diferencia absoluta:  {(p_treat - p_ctrl)*100:.2f} pp
  Lift relativo:        {(p_treat - p_ctrl) / p_ctrl * 100:.1f}%
  p-value:              {p_value:.6f}
  IC 95%:               [{ci_lower*100:.3f}, {ci_upper*100:.3f}] pp
  Power:                {actual_power:.1%}
""")

if p_value < 0.05 and ci_lower > 0:
    print("  RECOMENDACION: SHIP IT")
    print("  La diferencia es estadisticamente significativa y el IC no incluye 0.")
else:
    print("  RECOMENDACION: NO SHIP / seguir testeando")
    print("  La evidencia no es suficiente para concluir que el treatment es mejor.")

print("""
  CAVEATS:
  - Estos son datos sinteticos. En un experimento real, validar con
    metricas de negocio (revenue, retention, etc.) ademas de conversion.
  - El efecto puede variar por segmento (mobile vs desktop).
    Considerar lanzar solo para mobile si ahi es donde funciona mejor.
  - Monitorizar metricas secundarias post-lanzamiento.
  - Un A/B test no mide efectos a largo plazo (novelty effect).
""")

print("=" * 70)